## Extraction FastF1

In [ ]:

# import time
# import pandas as pd
# import numpy as np
# import fastf1
# from pathlib import Path
# from fastf1.req import RateLimitExceededError

# # ================= CONFIG (TON ARCHITECTURE) =================
# SEASONS = [2021, 2022, 2023, 2024, 2025]
# SESSION_TYPES = ["R", "Q"]  # Race + Qualifying
# # Racine = dossier FIL-ROUGE
# PROJECT_DIR = Path.cwd().parent

# # Dossier cache
# CACHE_DIR = PROJECT_DIR / "cache"
# CACHE_DIR.mkdir(parents=True, exist_ok=True)

# # Dossier data/raw
# DATA_RAW_DIR = PROJECT_DIR / "data" / "raw"
# DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

# # Activation cache FastF1
# fastf1.Cache.enable_cache(str(CACHE_DIR))

# OUT_FILE = DATA_RAW_DIR / "fastf1_kpis_R_Q_2021_2025.csv"


# # ================= HELPERS =================
# def safe_float(series: pd.Series) -> float:
#     """Retourne float si existe et non-NaN, sinon NaN."""
#     if series is None or len(series) == 0:
#         return np.nan
#     v = series.values[0]
#     return float(v) if pd.notna(v) else np.nan


# def extract_session_kpis(season: int, gp_name: str, session_type: str) -> pd.DataFrame:
#     """
#     Extrait des KPIs par pilote pour une session (R ou Q).
#     KPIs : best/avg/std lap times + max speed (fastest lap telemetry) + grid/finish/qualy position.
#     """
#     session = fastf1.get_session(season, gp_name, session_type)
#     session.load()

#     # laps (quicklaps = tours représentatifs)
#     laps = session.laps.pick_quicklaps()

#     # résultats officiels session (contient positions)
#     results_df = session.results

#     # infos event
#     round_no = int(session.event.RoundNumber) if pd.notna(session.event.RoundNumber) else np.nan
#     circuit = session.event.Location

#     output = []

#     # si pas de laps (cas rare), on retourne vide
#     if laps.empty:
#         return pd.DataFrame()

#     for driver in laps["Driver"].unique():
#         driver_laps = laps.pick_driver(driver)
#         if driver_laps.empty:
#             continue

#         best_lap = driver_laps["LapTime"].min()
#         avg_lap = driver_laps["LapTime"].mean()
#         std_lap = driver_laps["LapTime"].std()

#         # Telemetry: max speed sur le fastest lap du pilote
#         max_speed = np.nan
#         try:
#             fastest = driver_laps.pick_fastest()
#             telemetry = fastest.get_telemetry()
#             if telemetry is not None and "Speed" in telemetry.columns:
#                 max_speed = float(telemetry["Speed"].max())
#         except Exception:
#             pass

#         # Positions (on laisse en float pour supporter NaN)
#         grid_pos = safe_float(results_df.loc[results_df["Abbreviation"] == driver, "GridPosition"])
#         finish_pos = safe_float(results_df.loc[results_df["Abbreviation"] == driver, "Position"])
#         qualy_pos = safe_float(results_df.loc[results_df["Abbreviation"] == driver, "Position"]) if session_type == "Q" else np.nan

#         output.append({
#             "season": season,
#             "round": round_no,
#             "gp": gp_name,
#             "session": session_type,
#             "circuit": circuit,
#             "driverCode": driver,

#             "bestLapTime_sec": best_lap.total_seconds() if pd.notna(best_lap) else np.nan,
#             "avgLapTime_sec": avg_lap.total_seconds() if pd.notna(avg_lap) else np.nan,
#             "stdLapTime_sec": std_lap.total_seconds() if pd.notna(std_lap) else np.nan,
#             "maxSpeed_kmh": max_speed,

#             "gridPosition": grid_pos,
#             "finishPosition": finish_pos,
#             "qualyPosition": qualy_pos,
#         })

#     return pd.DataFrame(output)


# def load_done_keys(out_file: Path) -> set:
#     """Clé d'unité = (season, round, session)."""
#     if not out_file.exists():
#         return set()
#     df = pd.read_csv(out_file)
#     # si fichier vide ou colonnes absentes
#     if df.empty or not all(c in df.columns for c in ["season", "round", "session"]):
#         return set()
#     return set(zip(df["season"], df["round"], df["session"]))


# # ================= MAIN (INCREMENTAL + RESUME) =================
# done = load_done_keys(OUT_FILE)
# print(f"✅ Sessions déjà extraites: {len(done)}")

# for season in SEASONS:
#     schedule = fastf1.get_event_schedule(season)

#     # pour éviter une explosion d'appels, on itère proprement
#     for _, row in schedule.iterrows():
#         gp_name = row["EventName"]
#         round_no = int(row["RoundNumber"]) if pd.notna(row["RoundNumber"]) else None

#         for session_type in SESSION_TYPES:
#             key = (season, round_no, session_type)
#             if key in done:
#                 print(f"↩️ SKIP {season} round {round_no} {session_type} (déjà fait)")
#                 continue

#             tries = 0
#             while True:
#                 tries += 1
#                 try:
#                     df_gp = extract_session_kpis(season, gp_name, session_type=session_type)

#                     if df_gp.empty:
#                         print(f"⚠️ Vide: {season} round {round_no} {session_type} - {gp_name}")
#                         break

#                     # append incremental
#                     header = not OUT_FILE.exists()
#                     df_gp.to_csv(OUT_FILE, mode="a", index=False, header=header, encoding="utf-8")

#                     done.add(key)
#                     print(f"✔ Saved {season} round {round_no} {session_type} - {gp_name} (rows={len(df_gp)})")

#                     # throttle (réduit pression)
#                     time.sleep(3.0)
#                     break

#                 except RateLimitExceededError as e:
#                     wait_min = 20
#                     print(f"⏳ RATE LIMIT: pause {wait_min} min puis reprise... ({e})")
#                     time.sleep(wait_min * 60)

#                 except Exception as e:
#                     if tries >= 3:
#                         print(f"❌ FAIL {season} round {round_no} {session_type} - {gp_name}: {e}")
#                         break
#                     print(f"⚠️ RETRY {tries}/3 {season} round {round_no} {session_type} - {gp_name}: {e}")
#                     time.sleep(10)

# print(f"✅ Terminé. Fichier: {OUT_FILE}")


import time
import pandas as pd
import numpy as np
import fastf1
from pathlib import Path
from fastf1.req import RateLimitExceededError

# ================= CONFIG (TON ARCHITECTURE) =================
SEASONS = [2021, 2022, 2023, 2024, 2025]
SESSION_TYPES = ["Q", "R"]  # Qualif puis Race (ordre sans importance)

# Racine projet: si tu exécutes depuis /notebooks, .parent = Fil-rouge
PROJECT_DIR = Path.cwd().parent

CACHE_DIR = PROJECT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DATA_RAW_DIR = PROJECT_DIR / "data" / "raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

fastf1.Cache.enable_cache(str(CACHE_DIR))

OUT_FILE = DATA_RAW_DIR / "fastf1_kpis_R_Q_2021_2025.csv"

# ================= HELPERS =================
def safe_float(series: pd.Series) -> float:
    if series is None or len(series) == 0:
        return np.nan
    v = series.values[0]
    return float(v) if pd.notna(v) else np.nan


def clean_laps_for_kpis(laps: pd.DataFrame) -> pd.DataFrame:
    """
    Sécurise les laps pour KPI:
    - LapTime non null
    - quicklaps (exclut tours lents / SC en général)
    - exclut inlap/outlap si présent
    """
    if laps is None or laps.empty:
        return laps

    laps = laps[laps["LapTime"].notna()].copy()

    # quicklaps (bon filtre de base pour KPI)
    try:
        laps = laps.pick_quicklaps()
    except Exception:
        pass

    # enlever inlap/outlap si colonnes présentes
    for col in ["PitInTime", "PitOutTime"]:
        if col in laps.columns:
            laps = laps[laps[col].isna()]

    return laps


def extract_session_kpis(season: int, round_no: int, gp_name: str, session_type: str) -> pd.DataFrame:
    """
    KPIs par pilote pour une session (R ou Q):
    - best/avg/std LapTime (sec)
    - maxSpeed_kmh (telemetry fastest lap)
    - gridPosition (Race seulement)
    - finishPosition (Race seulement)
    - qualyPosition (Qualif seulement)
    """
    # ✅ IMPORTANT: utiliser round_no pour éviter les problèmes EventName / Pre-Season / alias
    session = fastf1.get_session(season, round_no, session_type)
    session.load()

    laps = clean_laps_for_kpis(session.laps)
    results_df = session.results

    # event info
    circuit = getattr(session.event, "Location", None) or session.event.get("Location", "")
    # gp_name vient du schedule (EventName), on garde pour lecture humain
    # round_no vient du schedule (fiable)

    if laps is None or laps.empty:
        return pd.DataFrame()

    output = []

    for driver in laps["Driver"].dropna().unique():
        driver_laps = laps.pick_driver(driver)
        if driver_laps.empty:
            continue

        best_lap = driver_laps["LapTime"].min()
        avg_lap = driver_laps["LapTime"].mean()
        std_lap = driver_laps["LapTime"].std()

        # ✅ max speed via télémétrie du fastest lap
        max_speed = np.nan
        try:
            fastest = driver_laps.pick_fastest()
            tel = fastest.get_telemetry()
            if tel is not None and "Speed" in tel.columns and not tel["Speed"].dropna().empty:
                max_speed = float(tel["Speed"].max())
        except Exception:
            pass

        # Positions (laisser en float pour supporter NaN)
        if session_type == "R":
            grid_pos = safe_float(results_df.loc[results_df["Abbreviation"] == driver, "GridPosition"])
            finish_pos = safe_float(results_df.loc[results_df["Abbreviation"] == driver, "Position"])
            qualy_pos = np.nan
        else:  # Q
            grid_pos = np.nan
            finish_pos = np.nan
            qualy_pos = safe_float(results_df.loc[results_df["Abbreviation"] == driver, "Position"])

        # Option utile: team (souvent dispo dans laps)
        team = np.nan
        if "Team" in driver_laps.columns:
            # 1 valeur typique
            t = driver_laps["Team"].dropna().unique()
            if len(t) > 0:
                team = t[0]

        output.append({
            "season": season,
            "round": int(round_no),
            "gp": gp_name,              # humain
            "session": session_type,
            "circuit": circuit,
            "driverCode": driver,
            "team": team,

            "bestLapTime_sec": best_lap.total_seconds() if pd.notna(best_lap) else np.nan,
            "avgLapTime_sec": avg_lap.total_seconds() if pd.notna(avg_lap) else np.nan,
            "stdLapTime_sec": std_lap.total_seconds() if pd.notna(std_lap) else np.nan,
            "maxSpeed_kmh": max_speed,

            "gridPosition": grid_pos,
            "finishPosition": finish_pos,
            "qualyPosition": qualy_pos,
        })

    return pd.DataFrame(output)


def load_done_keys(out_file: Path) -> set:
    """Clé unité = (season, round, session)"""
    if not out_file.exists():
        return set()
    df = pd.read_csv(out_file)
    if df.empty or not all(c in df.columns for c in ["season", "round", "session"]):
        return set()
    return set(zip(df["season"], df["round"], df["session"]))


# ================= MAIN (INCREMENTAL + RESUME) =================
expected_cols = [
    "season","round","gp","session","circuit","driverCode","team",
    "bestLapTime_sec","avgLapTime_sec","stdLapTime_sec","maxSpeed_kmh",
    "gridPosition","finishPosition","qualyPosition"
]

done = load_done_keys(OUT_FILE)
print(f"✅ Sessions déjà extraites: {len(done)}")

for season in SEASONS:
    schedule = fastf1.get_event_schedule(season)

    # ✅ Exclure Pre-Season / Testing si présents
    if "EventName" in schedule.columns:
        schedule = schedule[~schedule["EventName"].astype(str).str.contains("Pre-Season|Testing|Test", case=False, na=False)]

    for _, row in schedule.iterrows():
        gp_name = str(row.get("EventName", "")).strip()
        round_no = row.get("RoundNumber", None)

        if pd.isna(round_no):
            continue
        round_no = int(round_no)

        for session_type in SESSION_TYPES:
            key = (season, round_no, session_type)
            if key in done:
                print(f"↩️ SKIP {season} round {round_no} {session_type} ({gp_name})")
                continue

            tries = 0
            while True:
                tries += 1
                try:
                    df_gp = extract_session_kpis(season, round_no, gp_name, session_type=session_type)

                    if df_gp.empty:
                        print(f"⚠️ Vide: {season} round {round_no} {session_type} - {gp_name}")
                        break

                    # ⚠️ garantir colonnes dans le même ordre
                    df_gp = df_gp.reindex(columns=expected_cols)

                    header = not OUT_FILE.exists()
                    df_gp.to_csv(OUT_FILE, mode="a", index=False, header=header, encoding="utf-8")

                    done.add(key)
                    print(f"✔ Saved {season} round {round_no} {session_type} - {gp_name} (rows={len(df_gp)})")

                    # throttle
                    time.sleep(3.0)
                    break

                except RateLimitExceededError as e:
                    wait_min = 20
                    print(f"⏳ RATE LIMIT: pause {wait_min} min puis reprise... ({e})")
                    time.sleep(wait_min * 60)

                except Exception as e:
                    if tries >= 3:
                        print(f"❌ FAIL {season} round {round_no} {session_type} - {gp_name}: {e}")
                        break
                    print(f"⚠️ RETRY {tries}/3 {season} round {round_no} {session_type} - {gp_name}: {e}")
                    time.sleep(10)

print(f"✅ Terminé. Fichier: {OUT_FILE}")